# Single Student Prompt Experiments

This notebook runs a clean, single-student workflow for Experiments 1-4.
It is intentionally separated from the main notebook for clear thesis presentation.

In [ ]:
# 1) Configuration and Shared Utilities
import pandas as pd

from lib.experiment_utils import (
    create_client,
    load_best_attempts_df,
    select_target_student_id,
    get_student_data,
    build_strategies,
    run_experiment_rows,
    build_strategy_summary,
    save_results,
)

MODEL_ID = "gemini-2.5-flash"
RANDOM_SEED = 42
MIN_SUBMISSIONS = 5
TARGET_STUDENT_ID = None

client = create_client()
print(f"Environment ready. Using model: {MODEL_ID}")

Environment ready. Using model: gemini-2.5-flash


In [ ]:
# 2) Load Data and Select Student
best_attempts_df = load_best_attempts_df()
TARGET_STUDENT_ID = select_target_student_id(
    best_attempts_df=best_attempts_df,
    target_student_id=TARGET_STUDENT_ID,
    min_submissions=MIN_SUBMISSIONS,
)
student_data = get_student_data(best_attempts_df, TARGET_STUDENT_ID)

student_scores = best_attempts_df.groupby("SubjectID")["Score"].mean()
print(f"Selected student: {TARGET_STUDENT_ID}")
print(f"Average score: {student_scores.loc[TARGET_STUDENT_ID]:.2f}")
print(f"Total submissions: {len(student_data)}")
display(student_data.head())

Main table: 201,570 rows
CodeState table: 69,627 rows
Subject table: 381 rows
Joined dataset: 191,584 rows
Best attempts: 15,375 rows (372 students, 50 problems)
Selected student: 14355
Average score: 0.17
Total submissions: 5


,SubjectID,ProblemID,Order,ToolInstances,ServerTimestamp,ServerTimezone,CourseID,CourseSectionID,TermID,AssignmentID,...,EventType,Score,Compile.Result,CompileMessageType,CompileMessageData,EventID,ParentEventID,SourceLocation,Code,X-Grade
0,14355,13,27835,Java 8; CodeWorkout,2019-02-24T16:09:48,UTC,CS 1114,410.0,spring-2019,439.0,...,Run.Program,0.782609,None,None,None,13-55703,None,None,"public int caughtSpeeding(int speed, boolean i...",0.38
1,14355,24,141830,Java 8; CodeWorkout,2019-03-08T15:24:23,0,CS 1114,410.0,spring-2019,487.0,...,Run.Program,0.000000,None,None,None,24-53707,None,None,"public int blackjack(int a, int b)\r\n{\r\n ...",0.38
2,14355,40,174569,Java 8; CodeWorkout,2019-03-24T13:51:00,0,CS 1114,410.0,spring-2019,492.0,...,Run.Program,0.076923,None,None,None,40-32736,None,None,public String getSandwich(String str)\r\n{\r\n...,0.38
3,14355,41,62381,Java 8; CodeWorkout,2019-04-08T12:49:00,0,CS 1114,410.0,spring-2019,494.0,...,Run.Program,0.000000,None,None,None,41-11359,None,None,public int sum3(int[] nums)\r\n{\r\n return ...,0.38
4,14355,232,42303,Java 8; CodeWorkout,2019-02-24T16:35:06,UTC,CS 1114,410.0,spring-2019,439.0,...,Run.Program,0.000000,None,None,None,232-52239,None,None,"public String alarmClock(int day, boolean vaca...",0.38


In [ ]:
# 3) Strategies and Experiment Runner
strategies = build_strategies(
    focus_problem_ids=list(student_data["ProblemID"].unique())
)


def run_rows(rows_df: pd.DataFrame, sleep_seconds: float = 1.0) -> pd.DataFrame:
    return run_experiment_rows(
        rows_df=rows_df,
        client=client,
        model_id=MODEL_ID,
        strategies=strategies,
        sleep_seconds=sleep_seconds,
    )


print("Experiment runner configured.")

Experiment runner configured.


## Experiment 1: Single Assignment Analysis

In [ ]:
struggling_assignments = student_data[student_data["Score"] < 1.0]
target_assignment = struggling_assignments.iloc[0] if not struggling_assignments.empty else student_data.iloc[0]

print(f"Analyzing Problem {target_assignment['ProblemID']} (Score: {target_assignment['Score']:.2f})")
exp1_df = run_rows(pd.DataFrame([target_assignment]), sleep_seconds=0)

display_cols = ["SubjectID", "ProblemID", "Score"] + [c for c in exp1_df.columns if c.endswith("_Output")]
display(exp1_df[display_cols])

Analyzing Problem 13 (Score: 0.78)


,SubjectID,ProblemID,Score,Zero-Shot_Output,Few-Shot_Output,Chain-of-Thought_Output,Curriculum-Aware_Output
0,14355,13,0.782609,{'knowledge_gaps': ['Misinterpretation of prob...,{'knowledge_gaps': ['Misinterpretation of prob...,{'reasoning_chain': {'step1_behavior': 'The `c...,"{'student_analysis': [{'student_id': '14355', ..."


## Experiment 2: Multiple Assignments Analysis (5 Assignments)

In [ ]:
sample_size = min(5, len(student_data))
exp2_sample = student_data.sample(n=sample_size, random_state=RANDOM_SEED)

exp2_df = run_rows(exp2_sample, sleep_seconds=1)
display(exp2_df[["SubjectID", "ProblemID", "Score"] + [c for c in exp2_df.columns if c.endswith("_Output")]])

,SubjectID,ProblemID,Score,Zero-Shot_Output,Few-Shot_Output,Chain-of-Thought_Output,Curriculum-Aware_Output
0,14355,24,0.000000,{'knowledge_gaps': ['Method Return Values: The...,{'knowledge_gaps': ['Understanding method retu...,{'reasoning_chain': {'step1_behavior': 'The `b...,"{'student_analysis': [{'student_id': '14355', ..."
1,14355,232,0.000000,{'knowledge_gaps': ['Incorrect use of method r...,{'knowledge_gaps': ['Type mismatch: Attempting...,{'reasoning_chain': {'step1_behavior': 'The st...,"{'student_analysis': [{'student_id': '14355', ..."
2,14355,40,0.076923,{'knowledge_gaps': ['Problem Understanding and...,{'knowledge_gaps': ['Problem comprehension (no...,{'reasoning_chain': {'step1_behavior': 'The `g...,"{'student_analysis': [{'student_id': '14355', ..."
3,14355,13,0.782609,{'knowledge_gaps': ['Misinterpretation of prob...,{'knowledge_gaps': ['DRY Principle (Don't Repe...,{'reasoning_chain': {'step1_behavior': 'The co...,"{'student_analysis': [{'student_id': '14355', ..."
4,14355,41,0.000000,{'knowledge_gaps': ['Incorrect Java syntax for...,{'knowledge_gaps': ['Incorrect syntax for retu...,{'reasoning_chain': {'step1_behavior': 'The me...,"{'student_analysis': [{'student_id': '14355', ..."


## Experiment 3: Full History Analysis

In [ ]:
exp3_df = run_rows(student_data, sleep_seconds=1)
print("--- Experiment 3 Results ---")
display(exp3_df)

exp3_csv = f"single_student_{TARGET_STUDENT_ID}_exp3_results.csv"
save_results(exp3_df, exp3_csv)
print(f"Saved: {exp3_csv}")

--- Experiment 3 Results ---


,SubjectID,ProblemID,Score,Code,Zero-Shot_Output,Zero-Shot_TimeSec,Few-Shot_Output,Few-Shot_TimeSec,Chain-of-Thought_Output,Chain-of-Thought_TimeSec,Curriculum-Aware_Output,Curriculum-Aware_TimeSec
0,14355,13,0.782609,"public int caughtSpeeding(int speed, boolean i...",{'knowledge_gaps': ['Conditional Logic Optimiz...,16.424,{'knowledge_gaps': ['Misinterpretation of prob...,12.932,{'reasoning_chain': {'step1_behavior': 'The co...,14.725,"{'student_analysis': [{'student_id': '14355', ...",27.840
1,14355,24,0.000000,"public int blackjack(int a, int b)\r\n{\r\n ...",{'knowledge_gaps': ['Method Return Types: The ...,11.098,{'knowledge_gaps': ['Method Signature and Retu...,7.890,{'reasoning_chain': {'step1_behavior': 'The `b...,18.780,"{'student_analysis': [{'student_id': '14355', ...",10.488
2,14355,40,0.076923,public String getSandwich(String str)\r\n{\r\n...,{'knowledge_gaps': ['Understanding problem req...,8.498,{'knowledge_gaps': ['Problem decomposition and...,6.531,{'reasoning_chain': {'step1_behavior': 'The st...,9.599,"{'student_analysis': [{'student_id': '14355', ...",8.523
3,14355,41,0.000000,public int sum3(int[] nums)\r\n{\r\n return ...,{'knowledge_gaps': ['Basic Java syntax for arr...,6.891,{'knowledge_gaps': ['Fundamental Java syntax f...,6.828,{'reasoning_chain': {'step1_behavior': 'The `s...,8.191,"{'student_analysis': [{'student_id': '14355', ...",11.133
4,14355,232,0.000000,"public String alarmClock(int day, boolean vaca...",{'knowledge_gaps': ['**Method Return Types and...,10.682,{'knowledge_gaps': ['Recursion: Missing base c...,8.028,{'reasoning_chain': {'step1_behavior': 'The `a...,12.841,"{'student_analysis': [{'student_id': '14355', ...",15.532


Saved: single_student_14355_exp3_results.csv


In [ ]:
if 'exp3_df' not in locals() or exp3_df.empty:
    raise ValueError("Run Experiment 3 first.")

exp4_summary_df = build_strategy_summary(exp3_df, strategies)
display(exp4_summary_df)

exp4_csv = f"single_student_{TARGET_STUDENT_ID}_exp4_summary.csv"
save_results(exp4_summary_df, exp4_csv)
print(f"Saved: {exp4_csv}")

,Strategy,Valid_Responses,Total_Responses,Coverage_Pct,Avg_Time_Sec
1,Few-Shot,5,5,100.0,8.442
0,Zero-Shot,5,5,100.0,10.719
2,Chain-of-Thought,5,5,100.0,12.827
3,Curriculum-Aware,5,5,100.0,14.703


Saved: single_student_14355_exp4_summary.csv
